# Logistic Regression Model


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    BASE = "/content/drive/MyDrive/ColabNotebooks/ALDA"
    DATA_PATH = os.path.join(BASE, 'data', 'train.csv')
    RESULTS_DIR = os.path.join(BASE, 'econet', 'results')
    print("Running in Google Colab.")
except ImportError:
    print("Not running in Colab. Using local data path.")
    import os
    DATA_PATH = "data/train.csv"
    RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, average_precision_score, fbeta_score, confusion_matrix, precision_recall_curve, auc

# --- DATA LOADER ---
def load_econet_data(file_path):
    try:
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path, parse_dates=['Ob'])
        if 'Ob' in df.columns:
            df = df.sort_values(by=['Ob', 'Station']).reset_index(drop=True)
        return df
    except FileNotFoundError:
        print(f"File not found at {file_path}. Please check drive mounting.")
        return pd.DataFrame()

# --- FEATURE ENGINEERING ---
def compute_novel_features(df, time_col='Ob', station_col='Station', temp_col='value'):
    df_feat = df.copy()
    df_feat = df_feat.sort_values(by=[station_col, time_col]).reset_index(drop=True)

    # 1. temp_roc
    df_feat['temp_roc'] = df_feat.groupby(station_col)[temp_col].diff()
    df_feat['temp_roc'] = df_feat['temp_roc'].fillna(0)

    # 2. buddy_dev
    mean_temp_timestamp = df_feat.groupby(time_col)[temp_col].transform('mean')
    df_feat['buddy_dev'] = df_feat[temp_col] - mean_temp_timestamp
    df_feat['buddy_dev'] = df_feat['buddy_dev'].fillna(0)
    
    # 3. Time components
    if df_feat[time_col].dtype == 'object':
        df_feat[time_col] = pd.to_datetime(df_feat[time_col])
    df_feat['hour'] = df_feat[time_col].dt.hour
    df_feat['day_of_week'] = df_feat[time_col].dt.dayofweek
    df_feat['month'] = df_feat[time_col].dt.month

    # 4. QC Flag aggregates
    flag_cols = ['R_flag', 'I_flag', 'Z_flag', 'B_flag']
    existing_flags = [c for c in flag_cols if c in df_feat.columns]
    if existing_flags:
        active_flags = (df_feat[existing_flags] != 0).astype(int)
        df_feat['flag_sum'] = active_flags.sum(axis=1)
        df_feat['any_flag'] = (df_feat['flag_sum'] > 0).astype(int)
    else:
        df_feat['flag_sum'] = 0
        df_feat['any_flag'] = 0

    return df_feat

novel_feature_transformer = FunctionTransformer(compute_novel_features, kw_args={})
log1p_transformer = FunctionTransformer(np.log1p, validate=False)

# --- PIPELINE BUILDER ---
def get_preprocessing_pipeline(numeric_features=None, skewed_features=None, categorical_features=None, use_scaler=True):
    if numeric_features is None: 
        numeric_features = ['value', 'temp_roc', 'buddy_dev', 'R_flag', 'I_flag', 'Z_flag', 'B_flag', 'hour', 'day_of_week', 'month', 'flag_sum', 'any_flag']
    if skewed_features is None: skewed_features = []
    if categorical_features is None: categorical_features = ['Station', 'measure']

    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if use_scaler:
        num_steps.append(('scaler', StandardScaler()))
    numeric_transformer = Pipeline(steps=num_steps)

    skew_steps = [('imputer', SimpleImputer(strategy='median')), ('log1p', log1p_transformer)]
    if use_scaler:
        skew_steps.append(('scaler', StandardScaler()))
    skewed_transformer = Pipeline(steps=skew_steps)

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('skewed', skewed_transformer, skewed_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'
    )
    return preprocessor

# --- SPLITTING ---
def temporal_train_test_split(df, target_col='target', time_col='Ob', test_size=0.2):
    if df.empty:
        raise ValueError("DataFrame is empty.")
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)
    split_idx = int(len(df_sorted) * (1 - test_size))
    train_df = df_sorted.iloc[:split_idx]
    test_df = df_sorted.iloc[split_idx:]
    
    return train_df.drop(columns=[target_col]), test_df.drop(columns=[target_col]), train_df[target_col], test_df[target_col]

# --- EVALUATION METRICS SAVER ---
def save_evaluation_metrics(model_name, y_true, y_pred, y_proba):
    print(f"\n=== {model_name} Performance ===")
    out_dir = os.path.join(RESULTS_DIR, model_name.replace(' ', '_').lower())
    os.makedirs(out_dir, exist_ok=True)
    
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    pr_auc = average_precision_score(y_true, y_proba)
    
    metrics = {
        'F2': f2,
        'PR-AUC': pr_auc,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    }
    
    with open(os.path.join(out_dir, f'{model_name}_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)
        
    print(f"F2 Score: {f2:.4f}")
    print(f"PR-AUC:   {pr_auc:.4f}")
    print(f"Results saved to {out_dir}")
    return metrics


In [ ]:
# Load Data
df_raw = load_econet_data(DATA_PATH)
if not df_raw.empty:
    print(f"Raw shape: {df_raw.shape}")

    print("Engineering features (including buddy_dev & temp_roc)...")
    df_eng = compute_novel_features(df_raw)

    print("Splitting chronologically...")
    X_train, X_test, y_train, y_test = temporal_train_test_split(df_eng, target_col='target', time_col='Ob')
    print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
else:
    print("DataFrame empty, skipping splits.")



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

categorical_features = ['Station', 'measure']
enhanced_numeric = ['value', 'temp_roc', 'buddy_dev', 'R_flag', 'I_flag', 'Z_flag', 'B_flag', 'hour', 'day_of_week', 'month', 'flag_sum', 'any_flag']

if not df_raw.empty:
    preprocessor = get_preprocessing_pipeline(enhanced_numeric, [], categorical_features, use_scaler=True)
    lr_base = LogisticRegression(solver='saga', max_iter=100, random_state=42)

    pipeline = Pipeline([('preprocess', preprocessor), ('classifier', lr_base)])
    tscv = TimeSeriesSplit(n_splits=5)

    param_search = {'classifier__C': [0.001, 0.01, 0.1, 1, 10], 'classifier__penalty': ['l2']}
    search = RandomizedSearchCV(pipeline, param_search, n_iter=3, scoring={'f2': make_scorer(fbeta_score, beta=2, zero_division=0), 'pr_auc': 'average_precision'}, refit='f2', cv=tscv, random_state=42)

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]
    save_evaluation_metrics("Logistic_Regression", y_test, y_pred, y_proba)
    joblib.dump(best_model, os.path.join(RESULTS_DIR, 'logistic_regression', 'lr_model.pkl'))

    # Save the winning hyperparameters and CV score
    out_dir = os.path.join(RESULTS_DIR, 'logistic_regression')
    os.makedirs(out_dir, exist_ok=True)
    tuning_results = {
        'best_parameters': search.best_params_,
        'best_cv_f2': float(search.best_score_)
    }
    with open(os.path.join(out_dir, 'hyperparameters.json'), 'w') as f:
        json.dump(tuning_results, f, indent=4)

